# Kiểm tra dữ liệu ban đầu – Criteo Uplift Modeling Dataset v2

## Mục tiêu kiểm tra

Notebook này dùng để đọc và quan sát ban đầu bộ dữ liệu trước khi thực hiện tiền xử lý. Các nội dung chính gồm kiểm tra kích thước, cấu trúc cột và một vài dòng dữ liệu mẫu.

## Thiết lập thư viện và đường dẫn dữ liệu

In [13]:
import pandas as pd
DATA_PATH = r"D:\uplift_modelling\data\criteo-uplift-v2.1.csv"

## Đọc dữ liệu

In [14]:
df = pd.read_csv(DATA_PATH)

print(f"Kích thước dữ liệu: {df.shape}")
df.head()

Kích thước dữ liệu: (13979592, 16)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


## Xem nhanh cấu trúc dữ liệu

Sau khi đọc dữ liệu, kiểm tra tên cột, kiểu dữ liệu, số lượng giá trị thiếu và các thống kê mô tả cơ bản để hiểu cấu trúc ban đầu của tập dữ liệu.

In [15]:
print("Tên các cột:")
print(df.columns.tolist())

print("\nKiểu dữ liệu và số lượng giá trị không thiếu:")
df.info()

Tên các cột:
['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'treatment', 'conversion', 'visit', 'exposure']

Kiểu dữ liệu và số lượng giá trị không thiếu:
<class 'pandas.DataFrame'>
RangeIndex: 13979592 entries, 0 to 13979591
Data columns (total 16 columns):
 #   Column      Dtype  
---  ------      -----  
 0   f0          float64
 1   f1          float64
 2   f2          float64
 3   f3          float64
 4   f4          float64
 5   f5          float64
 6   f6          float64
 7   f7          float64
 8   f8          float64
 9   f9          float64
 10  f10         float64
 11  f11         float64
 12  treatment   int64  
 13  conversion  int64  
 14  visit       int64  
 15  exposure    int64  
dtypes: float64(12), int64(4)
memory usage: 1.7 GB


Phân bố nhóm nhận quảng cáo và kết quả


In [16]:
group_summary = (
    df.groupby("treatment")
      .agg(
          so_quan_sat=("treatment", "size"),
          ty_le_chuyen_doi=("conversion", "mean"),
          ty_le_truy_cap=("visit", "mean"),
          ty_le_tiep_xuc=("exposure", "mean")
      )
)

group_summary["ty_le_quan_sat"] = (
    group_summary["so_quan_sat"] / len(df)
)

display(
    group_summary.style.format({
        "so_quan_sat": "{:,.0f}",
        "ty_le_quan_sat": "{:.2%}",
        "ty_le_chuyen_doi": "{:.4%}",
        "ty_le_truy_cap": "{:.2%}",
        "ty_le_tiep_xuc": "{:.2%}"
    })
)


,so_quan_sat,ty_le_chuyen_doi,ty_le_truy_cap,ty_le_tiep_xuc,ty_le_quan_sat
treatment,,,,,
0,"2,096,937",0.1938%,3.82%,0.00%,15.00%
1,"11,882,655",0.3089%,4.85%,3.60%,85.00%


Kiểm tra chất lượng dữ liệu

In [17]:
# Kiểm tra giá trị thiếu
missing_summary = pd.DataFrame({
    "so_luong_thieu": df.isna().sum(),
    "ty_le_thieu": df.isna().mean()
})

display(missing_summary)

# Kiểm tra các giá trị hợp lệ của biến nhị phân
for col in ["treatment", "conversion", "visit", "exposure"]:
    print(f"{col}: {sorted(df[col].unique())}")

,so_luong_thieu,ty_le_thieu
f0,0,0.0
f1,0,0.0
f2,0,0.0
f3,0,0.0
f4,0,0.0
f5,0,0.0
f6,0,0.0
f7,0,0.0
f8,0,0.0
f9,0,0.0


treatment: [np.int64(0), np.int64(1)]
conversion: [np.int64(0), np.int64(1)]
visit: [np.int64(0), np.int64(1)]
exposure: [np.int64(0), np.int64(1)]


Thống kê mô tả các biến đặc trưng

In [18]:
feature_cols = [f"f{i}" for i in range(12)]

feature_summary = df[feature_cols].describe().T

display(
    feature_summary.style.format({
        "count": "{:,.0f}",
        "mean": "{:.4f}",
        "std": "{:.4f}",
        "min": "{:.4f}",
        "25%": "{:.4f}",
        "50%": "{:.4f}",
        "75%": "{:.4f}",
        "max": "{:.4f}"
    })
)

,count,mean,std,min,25%,50%,75%,max
f0,"13,979,592",19.6203,5.3775,12.6164,12.6164,21.9234,24.4365,26.7453
f1,"13,979,592",10.0700,0.1048,10.0597,10.0597,10.0597,10.0597,16.3442
f2,"13,979,592",8.4466,0.2993,8.2144,8.2144,8.2144,8.7233,9.0520
f3,"13,979,592",4.1789,1.3366,-8.3984,4.6799,4.6799,4.6799,4.6799
f4,"13,979,592",10.3388,0.3433,10.2805,10.2805,10.2805,10.2805,21.1235
f5,"13,979,592",4.0285,0.4311,-9.0119,4.1155,4.1155,4.1155,4.1155
f6,"13,979,592",-4.1554,4.5779,-31.4298,-6.6993,-2.4111,0.2944,0.2944
f7,"13,979,592",5.1018,1.2052,4.8338,4.8338,4.8338,4.8338,11.9984
f8,"13,979,592",3.9336,0.0567,3.6351,3.9108,3.9719,3.9719,3.9719
f9,"13,979,592",16.0276,7.0190,13.1901,13.1901,13.1901,13.1901,75.2950


Mức độ đa dạng của các biến đặc trưng

In [19]:
# Số lượng giá trị riêng biệt và tỷ lệ giá trị phổ biến nhất
feature_distribution = pd.DataFrame({
    "so_gia_tri_khac_nhau": df[feature_cols].nunique(),
    "ty_le_gia_tri_pho_bien_nhat": [
        df[col].value_counts(normalize=True).iloc[0]
        for col in feature_cols
    ]
})

display(
    feature_distribution.style.format({
        "so_gia_tri_khac_nhau": "{:,.0f}",
        "ty_le_gia_tri_pho_bien_nhat": "{:.2%}"
    })
)

,so_gia_tri_khac_nhau,ty_le_gia_tri_pho_bien_nhat
f0,"2,181,959",26.51%
f1,60,98.77%
f2,"2,051,900",52.24%
f3,552,81.86%
f4,260,95.66%
f5,132,94.70%
f6,"1,645",26.51%
f7,"622,143",94.70%
f8,"3,743",52.24%
f9,"1,594",79.50%


Kiểm tra bản ghi trùng lặp hoàn toàn

In [20]:
duplicate_count = df.duplicated().sum()

print(f"Số bản ghi trùng lặp hoàn toàn: {duplicate_count:,}")
print(f"Tỷ lệ trùng lặp: {duplicate_count / len(df):.6%}")

Số bản ghi trùng lặp hoàn toàn: 1,259,545
Tỷ lệ trùng lặp: 9.009884%


In [21]:
# In toan bo dong thuoc cac nhom ban ghi trung lap.
duplicate_rows = df[df.duplicated(keep=False)]
print(duplicate_rows)

                 f0         f1        f2        f3         f4        f5  \
1         12.616365  10.059654  9.002689  4.679882  10.280525  4.115453   
2         12.616365  10.059654  8.964775  4.679882  10.280525  4.115453   
5         12.616365  10.059654  8.904507  4.679882  10.280525  4.115453   
10        12.616365  10.059654  8.807791  4.679882  10.280525  4.115453   
11        12.616365  10.059654  8.808535  4.679882  10.280525  4.115453   
...             ...        ...       ...       ...        ...       ...   
13979562  26.740980  10.059654  8.214383  4.679882  10.280525  4.115453   
13979570  12.616365  10.059654  8.708704  4.679882  10.280525  4.115453   
13979576  14.377107  10.059654  8.214383  3.907662  10.280525  4.115453   
13979577  23.302016  10.059654  8.214383  4.679882  10.280525  4.115453   
13979591  12.616365  10.059654  8.214383  4.679882  10.280525  3.013064   

                f6        f7        f8         f9       f10       f11  \
1         0.294443  4.8338

Kiểm tra mức độ cân bằng đặc trưng giữa hai nhóm

In [22]:
import numpy as np

balance_stats = df.groupby("treatment")[feature_cols].agg(["mean", "std"])

control_mean = balance_stats.loc[0].xs("mean", level=1)
treatment_mean = balance_stats.loc[1].xs("mean", level=1)

control_std = balance_stats.loc[0].xs("std", level=1)
treatment_std = balance_stats.loc[1].xs("std", level=1)

smd = (treatment_mean - control_mean) / np.sqrt(
    (control_std**2 + treatment_std**2) / 2
)

balance_summary = pd.DataFrame({
    "trung_binh_nhom_doi_chung": control_mean,
    "trung_binh_nhom_nhan_quang_cao": treatment_mean,
    "SMD": smd,
    "SMD_tuyet_doi": smd.abs()
})

display(
    balance_summary.style.format({
        "trung_binh_nhom_doi_chung": "{:.4f}",
        "trung_binh_nhom_nhan_quang_cao": "{:.4f}",
        "SMD": "{:.4f}",
        "SMD_tuyet_doi": "{:.4f}"
    })
)

,trung_binh_nhom_doi_chung,trung_binh_nhom_nhan_quang_cao,SMD,SMD_tuyet_doi
f0,19.6517,19.6148,-0.0069,0.0069
f1,10.0679,10.0703,0.0240,0.0240
f2,8.4482,8.4463,-0.0062,0.0062
f3,4.2328,4.1694,-0.0488,0.0488
f4,10.3365,10.3392,0.0080,0.0080
f5,4.0393,4.0266,-0.0306,0.0306
f6,-3.9999,-4.1828,-0.0404,0.0404
f7,5.0803,5.1056,0.0213,0.0213
f8,3.9347,3.9334,-0.0224,0.0224
f9,15.8863,16.0526,0.0240,0.0240


Quy mô của bốn tổ hợp treatment và conversion

In [23]:
stratum_summary = (
    df.groupby(["treatment", "conversion"])
      .size()
      .reset_index(name="so_quan_sat")
)

stratum_summary["ty_le_toan_bo_du_lieu"] = (
    stratum_summary["so_quan_sat"] / len(df)
)

display(
    stratum_summary.style.format({
        "so_quan_sat": "{:,.0f}",
        "ty_le_toan_bo_du_lieu": "{:.4%}"
    })
)

,treatment,conversion,so_quan_sat,ty_le_toan_bo_du_lieu
0,0,0,"2,092,874",14.9709%
1,0,1,"4,063",0.0291%
2,1,0,"11,845,944",84.7374%
3,1,1,"36,711",0.2626%
